In [2]:
# =============================================================================
# 1. Conexión Spark y carga de /LND
# =============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, to_date, concat_ws, sha2

spark = (SparkSession.builder
    .appName("F1_Refinamiento")
    .enableHiveSupport()
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

LND_PATH = "ort/LND"
RFN_PATH = "ort/RFN"
MIN_YEAR, MAX_YEAR = 2000, 2025

TABLAS_LND = [
    "seasons", "drivers", "constructors", "circuits",
    "status", "races", "results", "constructor_standings",
]

def cargar_tabla(nombre):
    """Lee el JSON multilínea de una tabla desde /LND."""
    return spark.read.option("multiLine", True).json(f"{LND_PATH}/{nombre}.json")

# Un dict con todos los DataFrames crudos, indexado por nombre de tabla
DATAFRAMES = {nombre: cargar_tabla(nombre) for nombre in TABLAS_LND}

In [4]:
# =============================================================================
# 2. Helpers de refinamiento
# =============================================================================
def recorte(df):
    """Recorte temporal 2000-2025 sobre una columna 'season' ya casteada a int."""
    return df.filter((col("season") >= MIN_YEAR) & (col("season") <= MAX_YEAR))

def guardar_rfn(df, nombre):
    """Escribe un DataFrame refinado como Parquet en /RFN y reporta el conteo."""
    ruta = f"{RFN_PATH}/{nombre}"
    df.write.mode("overwrite").parquet(ruta)
    print(f"[RFN] {nombre:24s} -> {ruta}   ({df.count()} filas, {len(df.columns)} columnas)")
    return df

In [20]:
# =============================================================================
# 3. Dimensiones simples: seasons, drivers, constructors, status
# =============================================================================

# seasons: catálogo de temporadas (recorte 2000-2025)
df_seasons_rfn = recorte(
    DATAFRAMES["seasons"].select(col("season").cast("int").alias("season"))
).dropDuplicates(["season"]).orderBy("season")
guardar_rfn(df_seasons_rfn, "seasons")

# drivers: catálogo de pilotos (sin recorte; se descarta url)
df_drivers_rfn = DATAFRAMES["drivers"].select(
    "driverId", "code", "familyName", "givenName", "permanentNumber",
    to_date("dateOfBirth").alias("dateOfBirth"), "nationality",
).dropDuplicates(["driverId"])
guardar_rfn(df_drivers_rfn, "drivers")

# constructors: catálogo de escuderías (sin recorte; se descarta url)
df_constructors_rfn = DATAFRAMES["constructors"].select(
    "constructorId", "name", "nationality",
).dropDuplicates(["constructorId"])
guardar_rfn(df_constructors_rfn, "constructors")

# status: catálogo de estados finales. 'count' -> 'count_api' (informativo, NO se usa para DNF)
df_status_rfn = DATAFRAMES["status"].select(
    col("statusId").cast("int").alias("statusId"),
    "status",
).dropDuplicates(["statusId"])
guardar_rfn(df_status_rfn, "status")

[RFN] seasons                  -> ort/RFN/seasons   (26 filas, 1 columnas)
[RFN] drivers                  -> ort/RFN/drivers   (881 filas, 7 columnas)
[RFN] constructors             -> ort/RFN/constructors   (214 filas, 3 columnas)
[RFN] status                   -> ort/RFN/status   (136 filas, 2 columnas)


DataFrame[statusId: int, status: string]

In [19]:
# =============================================================================
# 4. circuits: se aplana Location; lat/long a double (usados en el mapa de folium)
# =============================================================================
df_circuits_rfn = DATAFRAMES["circuits"].select(
    "circuitId",
    "circuitName",
    col("Location.country").alias("country"),
    col("Location.locality").alias("locality"),
    col("Location.lat").cast("double").alias("latitude"),
    col("Location.long").cast("double").alias("longitude"),
).dropDuplicates(["circuitId"])
guardar_rfn(df_circuits_rfn, "circuits")

[RFN] circuits                 -> ort/RFN/circuits   (78 filas, 6 columnas)


DataFrame[circuitId: string, circuitName: string, country: string, locality: string, latitude: double, longitude: double]

In [17]:
# =============================================================================
# 5. races: se descarta el struct Circuit (se deja circuitId como FK), las
#    sesiones (prácticas/qualy/sprint) y url. Clave sintética raceId = sha2(season_round).
# =============================================================================
df_races_base = DATAFRAMES["races"].select(
    col("season").cast("int").alias("season"),
    col("round").cast("int").alias("round"),
    "raceName",
    to_date("date").alias("date"),
    col("Circuit.circuitId").alias("circuitId"),
)
df_races_rfn = (recorte(df_races_base)
    .withColumn("raceId", sha2(concat_ws("_", col("season"), col("round")), 256))
    .dropDuplicates(["raceId"]))
guardar_rfn(df_races_rfn, "races")

[RFN] races                    -> ort/RFN/races   (503 filas, 6 columnas)


DataFrame[season: int, round: int, raceName: string, date: date, circuitId: string, raceId: string]

In [21]:
# =============================================================================
# 6. results: explode del array Results -> una fila por (carrera, piloto).
#    Se conservan FKs (driverId, constructorId) + métricas; se descartan los structs
#    redundantes (Driver, Constructor) y los datos de carrera que ya están en 'races'.
#    raceId = mismo hash que races (FK); resultId sintético (PK del hecho).
# =============================================================================
df_results_base = DATAFRAMES["results"].select(
    "season", "round", explode("Results").alias("r")
).select(
    col("season").cast("int").alias("season"),
    col("round").cast("int").alias("round"),
    col("r.Driver.driverId").alias("driverId"),
    col("r.Constructor.constructorId").alias("constructorId"),
    col("r.grid").cast("int").alias("grid"),
    col("r.laps").cast("int").alias("laps"),
    col("r.number").cast("int").alias("number"),
    col("r.points").cast("double").alias("points"),
    col("r.position").cast("int").alias("position"),
    col("r.positionText").alias("positionText"),
    col("r.status").alias("status"),
)
df_results_rfn = (recorte(df_results_base)
    .withColumn("raceId", sha2(concat_ws("_", col("season"), col("round")), 256))
    .withColumn("resultId", sha2(
        concat_ws("_", col("season"), col("round"), col("driverId"), col("number")), 256)))
guardar_rfn(df_results_rfn, "results")

[RFN] results                  -> ort/RFN/results   (10550 filas, 13 columnas)


DataFrame[season: int, round: int, driverId: string, constructorId: string, grid: int, laps: int, number: int, points: double, position: int, positionText: string, status: string, raceId: string, resultId: string]

In [23]:
# =============================================================================
# 7. constructor_standings: explode del array. Snapshot de CIERRE de temporada
#    (no evolución por ronda: la API entrega un único registro por season,
#    correspondiente a la última ronda disputada — se verificó que ese round
#    coincide siempre con el total de carreras de la temporada en /LND races).
#    FK constructorId -> dim_constructor. raceId -> dim_race (misma clave
#    sintética que usa 'races': sha2(season_round)). Usado en la pregunta 5.
# =============================================================================
df_cs_base = DATAFRAMES["constructor_standings"].select(
    "season", "round", explode("ConstructorStandings").alias("cs")
).select(
    col("season").cast("int").alias("season"),
    col("round").cast("int").alias("totalRounds"),  # última ronda == total de carreras de la temporada
    col("cs.Constructor.constructorId").alias("constructorId"),
    col("cs.points").cast("double").alias("points"),
    col("cs.position").cast("int").alias("position"),        # null = sin puntos ("-") o excluido ("E"); no imputar
    col("cs.positionText").alias("positionText"),             # trazabilidad del caso null en position
    col("cs.wins").cast("int").alias("wins"),
)

df_cs_base = df_cs_base.withColumn(
    "raceId", sha2(concat_ws("_", col("season"), col("totalRounds")), 256)
).withColumn(
    "standingId", sha2(concat_ws("_", col("season"), col("constructorId")), 256)
)

df_cs_rfn = recorte(df_cs_base).select(
    "standingId", "raceId", "season", "totalRounds",
    "constructorId", "points", "position", "positionText", "wins"
)
guardar_rfn(df_cs_rfn, "constructor_standings")

[RFN] constructor_standings    -> ort/RFN/constructor_standings   (276 filas, 9 columnas)


DataFrame[standingId: string, raceId: string, season: int, totalRounds: int, constructorId: string, points: double, position: int, positionText: string, wins: int]

In [22]:
# =============================================================================
# 8. Verificación: releer /RFN y confirmar esquema, tipos y conteos post-recorte
# =============================================================================
for nombre in ["seasons", "drivers", "constructors", "status",
               "circuits", "races", "results", "constructor_standings"]:
    df = spark.read.parquet(f"{RFN_PATH}/{nombre}")
    print(f"\n===== /RFN/{nombre} - {df.count()} filas =====")
    df.printSchema()
    df.show(3, truncate=False)


===== /RFN/seasons - 26 filas =====
root
 |-- season: integer (nullable = true)

+------+
|season|
+------+
|2000  |
|2001  |
|2002  |
+------+
only showing top 3 rows


===== /RFN/drivers - 881 filas =====
root
 |-- driverId: string (nullable = true)
 |-- code: string (nullable = true)
 |-- familyName: string (nullable = true)
 |-- givenName: string (nullable = true)
 |-- permanentNumber: string (nullable = true)
 |-- dateOfBirth: date (nullable = true)
 |-- nationality: string (nullable = true)

+--------+----+----------+---------+---------------+-----------+-----------+
|driverId|code|familyName|givenName|permanentNumber|dateOfBirth|nationality|
+--------+----+----------+---------+---------------+-----------+-----------+
|Cannoc  |null|Cannon    |John     |null           |1933-06-21 |Canadian   |
|Changy  |null|de Changy |Alain    |null           |1922-02-05 |Belgian    |
|abate   |null|Abate     |Carlo    |null           |1932-07-10 |Italian    |
+--------+----+----------+--------